# 04 – Feature Engineering (BRFSS Diabetes)

Notebook này tập trung vào việc tạo và kiểm tra đặc trưng. Việc chia train/test và fit các phép biến đổi sẽ được thực hiện riêng trong notebook **05 – Pipeline & Split**.

**Schema:** 21 biến đầu vào và nhãn nhị phân `Diabetes_binary` (0/1). Đặc trưng bổ sung giữ nguyên 21 cột gốc: nhóm BMI theo ngưỡng WHO, tổng bệnh đồng mắc, điểm lối sống và tương tác Age × HighBP. Dữ liệu trùng được giữ lại để khớp với notebook nguồn và pipeline hiện có.

Chạy các ô từ trên xuống. Cần `pandas`, `numpy` và `scikit-learn` (đã có trong `requirements.txt`).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

TARGET = "Diabetes_binary"
RANDOM_STATE = 42
TEST_SIZE = 0.20
EXPECTED_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education",
    "Income", TARGET,
]

def find_project_root():
    # Notebook output directories are expected under <project>/notebooks/<stage>/.
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in candidates:
        if (base / "DATA").is_dir() or (base / "sources").is_dir():
            return base
    raise FileNotFoundError(
        "Không tìm thấy thư mục DATA/ hoặc sources/. Hãy mở notebook trong dự án "
        "hoặc chạy kernel từ một thư mục con của dự án."
    )

ROOT = find_project_root()
data_candidates = [
    ROOT / "DATA" / "raw" / "diabetes.csv",
    ROOT / "DATA" / "processed" / "diabetes_dedup.csv",
    ROOT / "sources" / "diabetes.csv",
]
DATA_PATH = next((p for p in data_candidates if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy diabetes.csv. Đặt dữ liệu tại DATA/raw/diabetes.csv, "
        "DATA/processed/diabetes_dedup.csv hoặc sources/diabetes.csv."
    )
df = pd.read_csv(DATA_PATH)
missing = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
if missing:
    raise ValueError(f"Thiếu cột theo schema BRFSS hiện tại: {missing}")
if not set(df[TARGET].dropna().unique()).issubset({0, 1}):
    raise ValueError(f"{TARGET} cần chứa nhãn nhị phân 0/1.")
print(f"Project: {ROOT}")
print(f"Dữ liệu: {DATA_PATH.relative_to(ROOT)}")
print(f"Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"Số ô thiếu: {df.isna().sum().sum():,}; dòng trùng: {df.duplicated().sum():,}")
print("Phân bố nhãn: ")
print(df[TARGET].value_counts(dropna=False).sort_index())
df.head()


Project: c:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Do_An_Cuoi_Ky
Dữ liệu: DATA\raw\diabetes.csv
Kích thước: 253,680 dòng × 22 cột
Số ô thiếu: 0; dòng trùng: 24,206
Phân bố nhãn: 
Diabetes_binary
0    218334
1     35346
Name: count, dtype: int64


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


## 1. Định nghĩa phép tạo đặc trưng

Transformer tương thích với scikit-learn và không ước lượng thống kê từ dữ liệu. Các quy tắc được áp dụng từng dòng; khi dùng trong notebook 05, transformer nằm bên trong pipeline để cả preprocessing và mô hình được fit trên train.

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin

class HealthFeatureEngineer(BaseEstimator, TransformerMixin):
    """Tạo đặc trưng từ các cột BRFSS hiện có, giữ nguyên các cột đầu vào."""
    def __init__(self, interaction=True):
        self.interaction = interaction

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        required = {"BMI", "HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack",
                    "PhysActivity", "Fruits", "Veggies", "Age"}
        missing = required - set(X.columns)
        if missing:
            raise ValueError(f"Thiếu cột cần thiết để tạo đặc trưng: {sorted(missing)}")
        return self

    def transform(self, X):
        out = X.copy()
        out["BMI_WHO_Group"] = pd.cut(
            out["BMI"], [-np.inf, 18.5, 25, 30, np.inf], labels=False, right=False
        ).astype(float)
        out["Comorbidity_Count"] = out[
            ["HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack"]
        ].sum(axis=1)
        out["Healthy_Lifestyle_Score"] = out[["PhysActivity", "Fruits", "Veggies"]].sum(axis=1)
        if self.interaction:
            out["Age_x_HighBP"] = out["Age"] * out["HighBP"]
        return out

    def get_feature_names_out(self, input_features=None):
        names = list(self.feature_names_in_) + [
            "BMI_WHO_Group", "Comorbidity_Count", "Healthy_Lifestyle_Score"
        ]
        if self.interaction:
            names.append("Age_x_HighBP")
        return np.asarray(names, dtype=object)


## 2. Tạo đặc trưng và kiểm tra

Ô này chỉ minh họa kết quả trên vài dòng để kiểm tra tên cột, giá trị và kích thước. Không dùng dữ liệu này để fit mô hình.

In [3]:
X = df.drop(columns=TARGET)
engineer = HealthFeatureEngineer()
engineered = engineer.fit_transform(X)
new_columns = [c for c in engineered.columns if c not in X.columns]
print(f"Số cột gốc: {X.shape[1]}; số đặc trưng thêm: {len(new_columns)}")
print("Đặc trưng mới:", new_columns)
display(engineered[new_columns].describe().T)
display(pd.concat([X.head(5), engineered[new_columns].head(5)], axis=1))


Số cột gốc: 21; số đặc trưng thêm: 4
Đặc trưng mới: ['BMI_WHO_Group', 'Comorbidity_Count', 'Healthy_Lifestyle_Score', 'Age_x_HighBP']


,count,mean,std,min,25%,50%,75%,max
BMI_WHO_Group,253680.0,2.049842,0.815440,0.0,1.0,2.0,3.0,3.0
Comorbidity_Count,253680.0,0.987878,0.968921,0.0,0.0,1.0,2.0,4.0
Healthy_Lifestyle_Score,253680.0,2.202219,0.880601,0.0,2.0,2.0,3.0,3.0
Age_x_HighBP,253680.0,3.966474,4.870205,0.0,0.0,0.0,9.0,13.0


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,PhysHlth,DiffWalk,Sex,Age,Education,Income,BMI_WHO_Group,Comorbidity_Count,Healthy_Lifestyle_Score,Age_x_HighBP
0,1,1,1,40,1,0,0,0,0,1,...,15,1,0,9,4,3,3.0,2,1,9
1,0,0,0,25,1,0,0,1,0,0,...,0,0,0,7,6,1,2.0,0,1,0
2,1,1,1,28,0,0,0,0,1,0,...,30,1,0,9,4,8,2.0,2,1,9
3,1,0,1,27,0,0,0,1,1,1,...,0,0,0,11,3,6,2.0,1,3,11
4,1,1,1,24,0,0,0,1,1,1,...,0,0,0,11,5,4,1.0,2,3,11


## Ghi chú

- `BMI_WHO_Group`: 0 = thiếu cân (<18,5), 1 = bình thường (18,5–<25), 2 = thừa cân (25–<30), 3 = béo phì (≥30).
- `Comorbidity_Count`: tổng `HighBP`, `HighChol`, `Stroke`, `HeartDiseaseorAttack` (0–4).
- `Healthy_Lifestyle_Score`: tổng `PhysActivity`, `Fruits`, `Veggies` (0–3); điểm lớn hơn nghĩa là nhiều hành vi tích cực hơn.
- `Age_x_HighBP`: tương tác giữa nhóm tuổi BRFSS và huyết áp cao.
- Mục tiêu không được đưa vào `X` và không bị biến đổi.